# 🏠 3D Home Scanner
촬영한 영상을 업로드하면 자동으로 3D 구조도를 만들어드립니다.

**순서:** 셀을 위에서 아래로 순서대로 실행하세요 (Shift+Enter)

In [ ]:
#@title ① 환경 설치 (처음 한 번만)
import subprocess, sys

# GPU 확인
result = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'], capture_output=True, text=True)
if result.returncode == 0:
    print(f'✅ GPU: {result.stdout.strip()}')
else:
    print('⚠️  GPU가 없습니다. 런타임 > 런타임 유형 변경 > T4 GPU 선택 후 다시 실행하세요.')
    sys.exit()

print('\n📦 NeRFStudio 설치 중... (3~5분 소요)')
subprocess.run(['pip', 'install', 'nerfstudio', '-q'], check=True)
subprocess.run(['pip', 'install', 'pycolmap', '-q'], check=True)
print('✅ 설치 완료!')

In [ ]:
#@title ② 영상 업로드
from google.colab import files
from pathlib import Path
import shutil, os

WORK_DIR = Path('/content/scan')
shutil.rmtree(WORK_DIR, ignore_errors=True)
WORK_DIR.mkdir(parents=True)

print('📁 영상 파일을 선택해주세요 (mp4, mov, avi)')
uploaded = files.upload()

video_path = None
for name, data in uploaded.items():
    video_path = WORK_DIR / name
    video_path.write_bytes(data)
    size_mb = len(data) / 1024 / 1024
    print(f'\n✅ 업로드 완료: {name} ({size_mb:.1f} MB)')
    break

if video_path is None:
    print('❌ 파일이 업로드되지 않았습니다.')

In [ ]:
#@title ③ 프레임 추출
import cv2
import numpy as np
from IPython.display import display, Image as IPImage

FRAME_INTERVAL_SEC = 0.5  #@param {type:"number"}
MAX_WIDTH = 1280          #@param {type:"integer"}

images_dir = WORK_DIR / 'images'
images_dir.mkdir(exist_ok=True)

cap = cv2.VideoCapture(str(video_path))
fps = cap.get(cv2.CAP_PROP_FPS)
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
duration = total_frames / fps
frame_step = max(1, int(fps * FRAME_INTERVAL_SEC))

print(f'🎬 영상 정보: {duration:.1f}초, {fps:.1f}fps, 총 {total_frames}프레임')
print(f'📸 {FRAME_INTERVAL_SEC}초마다 추출 예정: 약 {int(duration / FRAME_INTERVAL_SEC)}장')

saved, idx = 0, 0
preview_frames = []

while True:
    ret, frame = cap.read()
    if not ret:
        break
    if idx % frame_step == 0:
        h, w = frame.shape[:2]
        if w > MAX_WIDTH:
            scale = MAX_WIDTH / w
            frame = cv2.resize(frame, (MAX_WIDTH, int(h * scale)))
        path = images_dir / f'frame_{saved:04d}.jpg'
        cv2.imwrite(str(path), frame, [cv2.IMWRITE_JPEG_QUALITY, 90])
        if saved < 4:
            preview_frames.append(frame)
        saved += 1
    idx += 1

cap.release()
print(f'\n✅ {saved}장 추출 완료')

# 미리보기
if preview_frames:
    preview = np.hstack([cv2.resize(f, (320, 180)) for f in preview_frames])
    _, buf = cv2.imencode('.jpg', preview)
    display(IPImage(data=buf.tobytes()))

if saved < 20:
    print(f'\n⚠️  프레임이 {saved}장으로 너무 적습니다. 영상이 10초 이상이어야 합니다.')

In [ ]:
#@title ④ 카메라 포즈 추정 (COLMAP)
import subprocess

colmap_dir = WORK_DIR / 'colmap'
colmap_dir.mkdir(exist_ok=True)

print('📐 카메라 포즈 추정 중... (1~3분 소요)')

result = subprocess.run(
    [
        'ns-process-data', 'images',
        '--data', str(images_dir),
        '--output-dir', str(colmap_dir),
    ],
    capture_output=True, text=True
)

if result.returncode != 0:
    print('❌ 실패:\n', result.stdout[-1000:], result.stderr[-500:])
elif not (colmap_dir / 'transforms.json').exists():
    print('❌ transforms.json 없음. 영상이 너무 흔들렸거나 짧을 수 있습니다.')
    print(result.stdout[-800:])
else:
    import json
    tf = json.loads((colmap_dir / 'transforms.json').read_text())
    print(f'✅ 완료! {len(tf["frames"])}개 프레임 포즈 추정 성공')

In [ ]:
#@title ⑤ Gaussian Splatting 학습
import subprocess

train_dir = WORK_DIR / 'output'
NUM_ITERATIONS = 7000  #@param {type:"integer"}

print(f'🧊 Gaussian Splatting 학습 시작 ({NUM_ITERATIONS} iterations)...')
print('T4 GPU 기준 약 10~15분 소요됩니다.\n')

result = subprocess.run(
    [
        'ns-train', 'splatfacto',
        '--data', str(colmap_dir),
        '--output-dir', str(train_dir),
        f'--max-num-iterations={NUM_ITERATIONS}',
        '--viewer.quit-on-train-completion=True',
        '--pipeline.model.cull-alpha-thresh=0.005',
    ],
    capture_output=True, text=True
)

if result.returncode != 0:
    print('❌ 학습 실패:\n', result.stdout[-1000:], result.stderr[-500:])
else:
    print('✅ 학습 완료!')
    config_files = list(train_dir.glob('**/config.yml'))
    if config_files:
        CONFIG_PATH = config_files[-1]
        print(f'📄 Config: {CONFIG_PATH}')

In [ ]:
#@title ⑥ .splat 파일 export
import subprocess, shutil

export_dir = WORK_DIR / 'export'
export_dir.mkdir(exist_ok=True)

result = subprocess.run(
    [
        'ns-export', 'gaussian-splat',
        '--load-config', str(CONFIG_PATH),
        '--output-dir', str(export_dir),
    ],
    capture_output=True, text=True
)

splat_files = list(export_dir.glob('*.splat')) + list(export_dir.glob('**/*.splat'))
ply_files   = list(export_dir.glob('*.ply'))   + list(export_dir.glob('**/*.ply'))

SPLAT_PATH = None
if splat_files:
    SPLAT_PATH = splat_files[0]
    print(f'✅ .splat 파일: {SPLAT_PATH} ({SPLAT_PATH.stat().st_size/1024/1024:.1f} MB)')
elif ply_files:
    SPLAT_PATH = ply_files[0]
    print(f'✅ .ply 파일 (splat 대체): {SPLAT_PATH}')
else:
    print('❌ export 실패:', result.stdout[-500:], result.stderr[-300:])

In [ ]:
#@title ⑦ 3D 뷰어 (Colab 인라인)
import base64
from IPython.display import HTML

splat_b64 = base64.b64encode(SPLAT_PATH.read_bytes()).decode()
file_ext  = SPLAT_PATH.suffix.lower()

html = f"""
<div style="width:100%; height:600px; background:#08080f; border-radius:16px; overflow:hidden; position:relative;">
  <canvas id="c" style="width:100%;height:100%;display:block;"></canvas>
  <div id="info" style="position:absolute;top:16px;left:50%;transform:translateX(-50%);background:rgba(0,0,0,.6);color:#fff;padding:8px 16px;border-radius:20px;font-family:sans-serif;font-size:13px;">로딩 중…</div>
  <div style="position:absolute;bottom:16px;left:50%;transform:translateX(-50%);color:rgba(255,255,255,.4);font-size:11px;font-family:sans-serif;">드래그: 회전 &nbsp;|&nbsp; 휠: 확대/축소</div>
</div>
<script type="importmap">
{{"imports": {{"three": "https://unpkg.com/three@0.160.0/build/three.module.js",
               "three/addons/": "https://unpkg.com/three@0.160.0/examples/jsm/"}}}}
</script>
<script type="module">
import * as THREE from 'three';
import {{ OrbitControls }} from 'three/addons/controls/OrbitControls.js';
import {{ PLYLoader }} from 'three/addons/loaders/PLYLoader.js';

const canvas = document.getElementById('c');
const info   = document.getElementById('info');
const renderer = new THREE.WebGLRenderer({{canvas, antialias:true}});
renderer.setPixelRatio(window.devicePixelRatio);
renderer.setClearColor(0x08080f);

const scene  = new THREE.Scene();
const camera = new THREE.PerspectiveCamera(60, canvas.clientWidth/canvas.clientHeight, 0.001, 100);
camera.position.set(0, 0, 3);

const controls = new OrbitControls(camera, canvas);
controls.enableDamping = true;

function resize() {{
  const w = canvas.clientWidth, h = canvas.clientHeight;
  renderer.setSize(w, h, false);
  camera.aspect = w/h;
  camera.updateProjectionMatrix();
}}
resize();
window.addEventListener('resize', resize);

// base64 → ArrayBuffer
const b64 = '{splat_b64}';
const binary = atob(b64);
const bytes  = new Uint8Array(binary.length);
for (let i=0;i<binary.length;i++) bytes[i]=binary.charCodeAt(i);

const ext = '{file_ext}';
if (ext === '.ply') {{
  const loader = new PLYLoader();
  const geo = loader.parse(bytes.buffer);
  geo.computeVertexNormals();
  const mat = new THREE.PointsMaterial({{size:0.005, vertexColors:true}});
  if (!geo.attributes.color) {{
    mat.color.set(0x8888ff);
    mat.vertexColors = false;
  }}
  const pts = new THREE.Points(geo, mat);
  // 중심 이동
  geo.computeBoundingBox();
  const center = new THREE.Vector3();
  geo.boundingBox.getCenter(center);
  pts.position.sub(center);
  scene.add(pts);
  const size = new THREE.Vector3();
  geo.boundingBox.getSize(size);
  camera.position.set(0, 0, Math.max(...size.toArray()) * 1.5);
  controls.update();
  info.textContent = `✅ 포인트 ${{(geo.attributes.position.count/1000).toFixed(0)}}K개`;
}} else {{
  info.textContent = '.splat 뷰어 준비 중…';
  // Gaussian splat 렌더링 (antimatter15/splat 방식)
  loadSplat(bytes, scene, camera, renderer, controls, info);
}}

function animate() {{
  requestAnimationFrame(animate);
  controls.update();
  renderer.render(scene, camera);
}}
animate();

async function loadSplat(bytes, scene, camera, renderer, controls, info) {{
  // splat 파일 파싱: 각 가우시안 = 3(pos)+3(scale)+4(rot)+4(rgba) = 14 float32 + 1 alpha uint8
  // 실제 .splat 포맷: x,y,z (f32) + scale_x,y,z (f32) + r,g,b,a (u8) + rot_x,y,z,w (u8)
  const ITEM = 3*4 + 3*4 + 4 + 4;  // 32 bytes per splat
  const n = Math.floor(bytes.length / ITEM);
  info.textContent = `파싱 중… ${{(n/1000).toFixed(0)}}K splats`;

  const pos   = new Float32Array(n * 3);
  const color = new Float32Array(n * 3);
  const view  = new DataView(bytes.buffer);

  for (let i=0; i<n; i++) {{
    const base = i * ITEM;
    pos[i*3]   = view.getFloat32(base,    true);
    pos[i*3+1] = view.getFloat32(base+4,  true);
    pos[i*3+2] = view.getFloat32(base+8,  true);
    color[i*3]   = bytes[base+24] / 255;
    color[i*3+1] = bytes[base+25] / 255;
    color[i*3+2] = bytes[base+26] / 255;
  }}

  const geo = new THREE.BufferGeometry();
  geo.setAttribute('position', new THREE.BufferAttribute(pos, 3));
  geo.setAttribute('color',    new THREE.BufferAttribute(color, 3));
  geo.computeBoundingBox();

  const mat = new THREE.PointsMaterial({{size:0.008, vertexColors:true, sizeAttenuation:true}});
  const pts = new THREE.Points(geo, mat);

  const center = new THREE.Vector3();
  geo.boundingBox.getCenter(center);
  pts.position.sub(center);

  const size = new THREE.Vector3();
  geo.boundingBox.getSize(size);
  camera.position.set(0, 0, Math.max(...size.toArray()) * 1.2);
  controls.update();
  scene.add(pts);
  info.textContent = `✅ ${{(n/1000).toFixed(0)}}K Gaussians`;
}}
</script>
"""

display(HTML(html))

In [ ]:
#@title ⑧ (선택) 결과 파일 다운로드
from google.colab import files
files.download(str(SPLAT_PATH))
print('✅ 다운로드 시작됨')